# Suite FRM — the framework every language shares

What `Framework.Definitions` and `Framework.Errors` give every language, independent of how its nodes are held: the
entities a program declares and the scopes that find them (`Entity`, `Scope`, `Program`), checked here with plain
objects as nodes, since the port brings the languages over one stage at a time and the lookups that C and C++ and
TypeScript use (transparent scopes, using-directives, bases, qualified lookup, aliases) come back with their stages; and
the errors the languages raise.

In [ ]:
from mbse.Programs.Framework.Definitions import Entity, Program, Scope
from mbse.Programs.Framework.Errors import ParseError, PrintError, TranspileError
from support import raises


def entity(kind, name, parent, scope_kind=None):
    """An entity declared in `parent`, with a scope of its own where `scope_kind` is given."""
    made = parent.declare(Entity(kind, name, parent))
    if scope_kind is not None:
        made.scope = Scope(scope_kind, made, parent)
    return made

## FRM-01 · Names, qualified names and aliases

In [ ]:
root = Scope("namespace", None, None)
geo = entity("namespace", "geo", root, "namespace")
point = entity("class", "Point", geo.scope, "class")
unnamed = entity("namespace", None, root, "namespace")
assert point.qualified_name() == "geo::Point" and repr(point) == "<class geo::Point>"
assert repr(unnamed) == "<namespace (unnamed)>" and unnamed not in root.entities()
assert Entity("global", None, None).qualified_name() == ""
alias = entity("alias", "P", root)
alias.target = point
again = entity("alias", "Q", root)
again.target = alias
assert again.resolved() is point and point.resolved() is point
loop = entity("alias", "L", root)
loop.target = loop
assert loop.resolved() is loop  # an alias of itself stops
dotted = Scope("module", None, None, separator=".")
member = entity("function", "area", entity("class", "Shape", dotted, "class").scope)
assert member.qualified_name() == "Shape.area"
assert root.declare(point, "Alias") is point and root.declare(point, "Alias") is point
assert root.lookup("Alias") == [point]
assert repr(geo.scope) == "<namespace scope geo>" and repr(root) == "<namespace scope>"


## FRM-02 · Transparent scopes, using-directives and bases

In [ ]:
root = Scope("namespace", None, None)
inline = Scope("namespace", None, root)
hidden = entity("variable", "hidden", inline)
root.transparent.append(inline)
inline.transparent.append(root)  # a cycle of transparent scopes is walked once
assert root.lookup("hidden") == [hidden]
std = entity("namespace", "std", root, "namespace")
vector = entity("class", "vector", std.scope)
block = Scope("block", None, root)
block.using.append(std.scope)
assert block.resolve("vector") == [vector] and block.resolve("nothing") == []
base = entity("class", "Base", root, "class")
size = entity("function", "size", base.scope)
derived = entity("class", "Derived", root, "class")
derived.scope.bases.append(base.scope)
assert derived.scope.lookup("size") == [size] and derived.scope.lookup("nothing") == []


## FRM-03 · Qualified lookup, from a scope or from the global scope

In [ ]:
root = Scope("namespace", None, None)
geo = entity("namespace", "geo", root, "namespace")
v1 = entity("namespace", "v1", geo.scope, "namespace")
point = entity("class", "Point", v1.scope)
alias = entity("alias", "g", root)
alias.target = geo
inner = Scope("block", None, v1.scope)
assert inner.qualified(["geo", "v1", "Point"]) == [point]
assert inner.qualified(["g", "v1", "Point"]) == [point]  # through an alias of a namespace
assert inner.qualified(["Point"]) == [point]
assert inner.qualified(["geo", "v1", "Point"], from_global=True) == [point]
assert inner.qualified([]) == [] and inner.qualified(["Point", "x"]) == []  # a class with no scope qualifies nothing
assert inner.root() is root

program = Program(root)
node = object()
program.located(node, inner)
assert program.lookup("::geo::v1::Point") == [point] and program.lookup("geo::v1::Point", at=node) == [point]
assert program.lookup("Point", at=object()) == []  # a node outside the program: the global scope
program.declares(node, point)
assert program.entity_of(node) is point and program.scope_of(node) is inner and program.add(point) is point
assert list(program.entities()) == [point] and program.macros.kind == "macros"


## FRM-04 · The errors languages raise

In [ ]:
error = ParseError("unexpected token", 3, 7)
assert (error.line, error.column) == (3, 7) and isinstance(error, ValueError)
assert issubclass(PrintError, ValueError)
assert str(TranspileError("no counterpart", "body[0].value")) == "body[0].value: no counterpart"
assert str(TranspileError("outside the subset", "")) == "outside the subset"
assert TranspileError("x", "a.b").path == "a.b"
